In [ ]:
import pandas as pd
import altair as alt
from statannotations.stats.StatTest import StatTest
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator
import itertools

# Hardcoded file paths

In [ ]:
sge_file = './Data/final_tables/supplementary_file_1_BARD1_SGE_final_table.xlsx'

thermompnn_files = {
    'RING':'./Data/extra_data/ThermoMPNN_data/BARD1_RING.csv',
    'ARD':'./Data/extra_data/ThermoMPNN_data/BARD1_ARD.csv', #need to drop first 3 residues in the CSV
    'BRCT':'./Data/extra_data/ThermoMPNN_data/BARD1_BRCT.csv'
}

thermompnn_offsets ={
    'RING':26,
    'ARD': -2,
    'BRCT': 568
}

In [ ]:
def aa_spliter(df,aa_mut_col='amino_acid_change', ref_aa_col='ref_aa', aa_pos_col='aa_pos', alt_aa_col='alt_aa'):
    df[ref_aa_col] = df[aa_mut_col].transform(lambda x: x[0])
    df[alt_aa_col] = df[aa_mut_col].transform(lambda x: x[-1])
    df[aa_pos_col]=df[aa_mut_col].transform(lambda x: int(x[1:-1]))

    return df

# Dataframe processing and merging

## SGE df processing

In [ ]:
raw_sge_df = pd.read_excel(sge_file, sheet_name='scores')

sge_df = raw_sge_df[(raw_sge_df['amino_acid_change']!='---') & (raw_sge_df['var_type']=='snv')].copy()
sge_df =sge_df[['pos', 'ref', 'alt', 'exon', 'target', 'consequence', 'score', 'functional_consequence', 'amino_acid_change', 'pos_id', 'RNAscore', 'RNA_consequence']]
sge_df = aa_spliter(sge_df)

sge_df = sge_df[~sge_df['consequence'].isin(['stop_lost', 'stop_gained'])]

## ThermoMPNN processing

In [ ]:
domains = list(thermompnn_files.keys())

thermompnn_dfs = []

for domain in domains:
    thermompnn_df = pd.read_csv(thermompnn_files[domain])

    if domain =='ARD':
        thermompnn_df = thermompnn_df[thermompnn_df['pos']>100].copy()

    thermompnn_df['pos'] = thermompnn_df['pos'] + thermompnn_offsets[domain]
    thermompnn_df['amino_acid_change'] = thermompnn_df['wtAA'] + thermompnn_df['pos'].astype(str) + thermompnn_df['mutAA']

    thermompnn_df = thermompnn_df.rename(columns={'pos': 'aa_pos',
                                                    'wtAA': 'ref_aa',
                                                    'mutAA': 'alt_aa',
                                                    'ddG (kcal/mol)': 'ddG'})
    
    thermompnn_df['domain'] = domain
    thermompnn_df = thermompnn_df[['domain', 'amino_acid_change', 'ref_aa', 'alt_aa', 'aa_pos', 'ddG']]
    thermompnn_dfs.append(thermompnn_df)



final_thermompnn_df = pd.concat(thermompnn_dfs)
final_thermompnn_df

## Merge dataframes

In [ ]:
final_df = pd.merge(sge_df, final_thermompnn_df, on=['amino_acid_change', 'ref_aa', 'alt_aa', 'aa_pos'], how='inner')

final_df

# ThermomPNN vs. Fitness Score scatter

In [ ]:
scatter = alt.Chart(final_df[final_df['consequence'].isin(['missense_variant', 'synonymous_variant'])]).mark_circle().encode(
    x='score:Q',
    y='ddG:Q'
)

scatter.display()

# % LoF by ddG

In [ ]:
def thermompnn_scorer(df, threshold=None, bpdel=False):
    
    if threshold is None:
        normal_ddG = df[
            (df['functional_consequence'] == 'functionally_normal') &
            (df['consequence'] == 'missense_variant')
        ]['ddG']

        median_normal_ddG = normal_ddG.median()
        mad_normal_ddG    = (normal_ddG - median_normal_ddG).abs().median()
        mad_scaled        = 1.4826 * mad_normal_ddG

        threshold = median_normal_ddG + 2 * mad_scaled
        print(f' Median ddG: {median_normal_ddG:.3f}  |  MAD: {mad_normal_ddG:.3f}  |  Scaled MAD: {mad_scaled:.3f}')
        print(f' Threshold (median + 2 × scaled MAD): {threshold:.3f}')
    else:
        print(f' The threshold for "destabilizing" ddG is {threshold}')

    df['ddG_class'] = 'normal'
    df.loc[df['ddG'] >= threshold, 'ddG_class'] = 'destabilizing'

    if bpdel:
        print(df[df['functional_consequence'] == 'functionally_abnormal']
              .value_counts('ddG_class').reset_index())
        print(df[df['functional_consequence'] == 'functionally_abnormal']
              .value_counts('ddG_class', normalize=True).reset_index())
    else:
        lof_mis = df[
            (df['functional_consequence'] == 'functionally_abnormal') &
            (df['consequence'] == 'missense_variant') &
            (df['RNA_consequence'] == 'normal')
        ]
        print(lof_mis.value_counts('ddG_class').reset_index())
        print(lof_mis.value_counts('ddG_class', normalize=True).reset_index())

    return df, threshold

In [ ]:
final_df, ddG_threshold = thermompnn_scorer(final_df)

# Normal vs. LoF ddG comparison plots

## Violin plot

In [ ]:
missense_only = final_df[
    (final_df['consequence'] == 'missense_variant') &
    (final_df['functional_consequence'].isin(['functionally_normal', 'functionally_abnormal'])) &
    (final_df['RNA_consequence'] == 'normal')
].copy()

order = ['functionally_normal', 'Interacting LoF', 'Non-Interacting LoF']

missense_only['residue_class'] = missense_only['functional_consequence']
missense_only.loc[(missense_only['residue_class']=='functionally_abnormal') & 
                  (missense_only['aa_pos'].isin([712, 715, 467, 500, 429, 458, 462, 470, 705, 50, 66, 68, 71, 74, 83, 86])
), 'residue_class'] = 'Interacting LoF'

missense_only.loc[missense_only['residue_class']=='functionally_abnormal', 'residue_class'] = 'Non-Interacting LoF'

fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(
    data=missense_only,
    x='residue_class',
    y='ddG',
    order=order,
    ax=ax
)

ax.set_xlabel('Functional Consequence')
ax.set_ylabel('ddG (kcal/mol)')
plt.tight_layout()
plt.show()

## Cumulator plot

In [ ]:
missub = missense_only.sort_values(by="ddG")
gbobj = missub.groupby('residue_class')['ddG']
missub["cumulative_percentage"] = (gbobj.cumcount()  + 1)/ gbobj.transform('count') * 100.0


# draw the plot
chart = alt.Chart(missub, height=300, width=300).mark_line(size=5).encode(
    x=alt.X('ddG:Q', title='ddG').axis(labelFlush=False),
    y=alt.Y('cumulative_percentage:Q', title='Cumulative percentage of missense SNVs'),
    color = alt.Color('residue_class:N')
)

chart.display()

# Heatmap visualization

In [ ]:
# --- Global font: Arial for all matplotlib/seaborn and Altair plots ---
plt.rcParams['font.family'] = 'Arial'

@alt.theme.register('arial', enable=True)
def arial_theme():
    return alt.theme.ThemeConfig({'config': {'font': 'Arial'}})

# Universal font sizes — change these to resize all axes and legend labels at once
x_label_size = 22      # Font size for x-axis tick labels
x_title_size = 24      # Font size for x-axis titles
y_label_size = 22      # Font size for y-axis tick labels
y_title_size = 24      # Font size for y-axis titles
legend_label_size = 20 # Font size for legend item labels
legend_title_size = 22 # Font size for legend titles

def heatmap(df, score_col='score', score_name='Fitness Score', map_domain=[-0.2,0], reverse_colors=True):
    order = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y', 'Stop', 'Min.', 'Mean']

    height_per_category = 20

    #df = df.loc[~(df['AAsub'].isin(['*778*']))] #Gets rid of stop-loss variant
    
    map = alt.Chart(df).mark_rect().encode(
        x = alt.X('aa_pos:Q',
                  title = '',
                  axis = alt.Axis(values=list(range(0, 778, 25))),
                  scale = alt.Scale(domain = [0,778], zero=False),
                  bin = alt.Bin(maxbins = 778, minstep = 1)
                 ),
        y = alt.Y('alt_aa',
                  title = 'Amino Acid Substitution',
                  axis = alt.Axis(
                      labelFontSize = y_label_size,
                      titleFontSize = y_title_size
                  ),
                 sort = order),
        color = alt.Color(score_col, 
                          title = score_name,
                          scale = alt.Scale(
                              domain = map_domain,
                              clamp = True,
                              scheme = 'bluepurple',
                              reverse = reverse_colors
                          ),
                          legend = alt.Legend(
                              titleFontSize = legend_title_size,
                              labelFontSize = legend_label_size
                          )
                         ),
        tooltip=['amino_acid_change','score', 'ddG']
    ).properties(
        height = height_per_category * len(order), 
        width = 1450
    )

    return map

In [ ]:
def protein_cartoon(width=1450):
    """Domain and secondary-structure cartoon strip for BARD1.

    Produces a chart with the same x domain ([0, 778]) and width as the heatmaps
    so it aligns when vconcat-ed above them.

    Row 1 (y 0→1): secondary structure elements — blue=α-helix, green=β-sheet, white=loop
    Row 2 (y 1.15→2.15): domain labels (RING / ARD / BRCT)
    """

    # Secondary structure coordinates (from solved structures: 1JM7, 3C5R, 3FA2)
    ss_x = [
        # RING (1JM7)
        26,  34,  48,  61, 63,  68, 70,  74,  80,  97, 117,
        # ARD (3C5R)
        425, 430, 439, 441, 450, 463, 471, 473, 483, 497, 504, 507, 516, 529, 534, 536,
        # BRCT (3FA2)
        568, 571, 574, 578, 592, 595, 597, 606, 608, 617, 626, 629, 631, 643,
        655, 666, 676, 679, 687, 698, 700, 702, 712, 717, 735, 739, 750, 752, 755, 760, 770,
    ]
    ss_x2 = [
        # RING
        34,  48,  61,  63, 68,  70, 74,  80,  97, 117, 122,
        # ARD
        430, 439, 441, 450, 463, 471, 473, 483, 497, 504, 507, 516, 529, 534, 536, 545,
        # BRCT
        571, 574, 578, 592, 595, 597, 606, 608, 617, 626, 629, 631, 643, 655,
        666, 676, 679, 687, 698, 700, 702, 712, 717, 735, 739, 750, 752, 755, 760, 770, 777,
    ]
    ss_colors = [
        # RING
        'white', 'blue', 'white', 'green', 'white', 'green', 'white', 'blue', 'white', 'blue', 'white',
        # ARD
        'white', 'blue', 'white', 'blue', 'white', 'blue', 'white', 'blue',
        'white', 'blue', 'white', 'blue', 'white', 'blue', 'white', 'blue',
        # BRCT
        'white', 'green', 'white', 'blue', 'white', 'green', 'white', 'green',
        'white', 'blue', 'white', 'green', 'blue', 'white', 'blue', 'white',
        'green', 'white', 'blue', 'white', 'green', 'white', 'blue', 'white',
        'green', 'white', 'green', 'white', 'green', 'blue', 'white',
    ]

    ss_df = pd.DataFrame({'x': ss_x, 'x2': ss_x2, 'y': 0, 'y2': 1, 'color': ss_colors})

    dom_df = pd.DataFrame({
        'x':     [26,       425,      568],
        'x2':    [122,      545,      777],
        'y':     [1.15,     1.15,     1.15],
        'y2':    [2.15,     2.15,     2.15],
        'color': ['#B9DBF4', '#C8DBC8', '#F6BF93'],
        'label': ['RING',   'ARD',    'BRCT'],
    })

    x_scale = alt.Scale(domain=[0, 778])
    y_scale = alt.Scale(domain=[-0.15, 2.3])
    no_axis = alt.Axis(labels=False, ticks=False, domain=False, title=None, grid=False)

    ss_rects = alt.Chart(ss_df).mark_rect(stroke='black', strokeWidth=1.5).encode(
        x =alt.X('x:Q',  scale=x_scale, axis=no_axis),
        x2='x2:Q',
        y =alt.Y('y:Q',  scale=y_scale, axis=no_axis),
        y2='y2:Q',
        color=alt.Color('color:N', scale=None, legend=None),
    )

    dom_rects = alt.Chart(dom_df).mark_rect(stroke='black', strokeWidth=2, opacity=0.85).encode(
        x =alt.X('x:Q',  scale=x_scale, axis=no_axis),
        x2='x2:Q',
        y =alt.Y('y:Q',  scale=y_scale, axis=no_axis),
        y2='y2:Q',
        color=alt.Color('color:N', scale=None, legend=None),
    )

    dom_text = alt.Chart(dom_df).mark_text(
        fontWeight='bold', fontSize=16, color='black',
        align='center', baseline='middle',
    ).encode(
        x   =alt.X('x_mid:Q', scale=x_scale, axis=no_axis),
        y   =alt.Y('y_mid:Q', scale=y_scale, axis=no_axis),
        text='label:N',
    ).transform_calculate(
        x_mid='(datum.x + datum.x2) / 2',
        y_mid='(datum.y + datum.y2) / 2',
    )

    return alt.layer(ss_rects, dom_rects, dom_text).properties(width=width, height=60)

In [ ]:
sge_map = heatmap(final_df)

In [ ]:
ddg_map = heatmap(final_df, score_col='ddG', score_name='ddG', map_domain=[0, 3], reverse_colors=False)

In [ ]:
final_map = (sge_map & ddg_map).resolve_scale(color='independent').interactive()

final_map

In [ ]:
lof_only = final_df[final_df['functional_consequence']=='functionally_abnormal']
lof_ddG_map = heatmap(lof_only, score_col='ddG', score_name='ddG', map_domain=[0, 2], reverse_colors=False)

full_lof_map = (sge_map & lof_ddG_map).interactive().resolve_scale(color='independent')

full_lof_map.display()

# By secondary structural element

In [ ]:
def annotate_ss(df):
    """Add ss_element column (α-Helix / β-Sheet / Loop) using the same half-open
    [start, end) intervals that drive the domain cartoon rectangles."""

    ss_intervals = {
        'RING': [                              # 1JM7
            (26,  34,  'Loop'),
            (34,  48,  'α-Helix'),
            (48,  61,  'Loop'),
            (61,  63,  'β-Sheet'),
            (63,  68,  'Loop'),
            (68,  70,  'β-Sheet'),
            (70,  74,  'Loop'),
            (74,  80,  'α-Helix'),
            (80,  97,  'Loop'),
            (97,  117, 'α-Helix'),
            (117, 123, 'Loop'),
        ],
        'ARD': [                               # 3C5R
            (425, 430, 'Loop'),
            (430, 439, 'α-Helix'),
            (439, 441, 'Loop'),
            (441, 450, 'α-Helix'),
            (450, 463, 'Loop'),
            (463, 471, 'α-Helix'),
            (471, 473, 'Loop'),
            (473, 483, 'α-Helix'),
            (483, 497, 'Loop'),
            (497, 504, 'α-Helix'),
            (504, 507, 'Loop'),
            (507, 516, 'α-Helix'),
            (516, 529, 'Loop'),
            (529, 534, 'α-Helix'),
            (534, 536, 'Loop'),
            (536, 546, 'α-Helix'),
        ],
        'BRCT': [                              # 3FA2
            (568, 571, 'Loop'),
            (571, 574, 'β-Sheet'),
            (574, 578, 'Loop'),
            (578, 592, 'α-Helix'),
            (592, 595, 'Loop'),
            (595, 597, 'β-Sheet'),
            (597, 606, 'Loop'),
            (606, 608, 'β-Sheet'),
            (608, 617, 'Loop'),
            (617, 626, 'α-Helix'),
            (626, 629, 'Loop'),
            (629, 631, 'β-Sheet'),
            (631, 643, 'α-Helix'),
            (643, 655, 'Loop'),
            (655, 666, 'α-Helix'),
            (666, 676, 'Loop'),
            (676, 679, 'β-Sheet'),
            (679, 687, 'Loop'),
            (687, 698, 'α-Helix'),
            (698, 700, 'Loop'),
            (700, 702, 'β-Sheet'),
            (702, 712, 'Loop'),
            (712, 717, 'α-Helix'),
            (717, 735, 'Loop'),
            (735, 739, 'β-Sheet'),
            (739, 750, 'Loop'),
            (750, 752, 'β-Sheet'),
            (752, 755, 'Loop'),
            (755, 760, 'β-Sheet'),
            (760, 770, 'α-Helix'),
            (770, 778, 'Loop'),
        ],
    }

    def _label(row):
        for start, end, label in ss_intervals.get(row['domain'], []):
            if start <= row['aa_pos'] < end:
                return label
        return 'Loop'

    df = df.copy()
    df['ss_element'] = df.apply(_label, axis=1)
    return df

In [ ]:
lof_ss = annotate_ss(
    final_df[
        (final_df['consequence'] == 'missense_variant') &
        (final_df['functional_consequence'] == 'functionally_abnormal') &
        (final_df['RNA_consequence'] == 'normal')
    ]
)

ss_order   = ['α-Helix', 'β-Sheet', 'Loop']
ss_palette = {'α-Helix': '#4472C4', 'β-Sheet': '#70AD47', 'Loop': '#BFBFBF'}

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, domain in zip(axes, ['RING', 'ARD', 'BRCT']):
    dom = lof_ss[lof_ss['domain'] == domain]
    present = [s for s in ss_order if s in dom['ss_element'].unique()]

    sns.violinplot(
        data=dom, x='ss_element', y='ddG',
        order=present, palette=ss_palette,
        inner='box', cut=0, ax=ax,
    )
    sns.stripplot(
        data=dom, x='ss_element', y='ddG',
        order=present, color='black',
        size=2.5, alpha=0.5, jitter=True, ax=ax,
    )

    ax.set_title(domain, fontsize=16, fontweight='bold')
    ax.set_xlabel('Secondary Structure Element', fontsize=12)
    ax.set_ylabel('ddG (kcal/mol)' if domain == 'RING' else '', fontsize=12)
    ax.axhline(1.665, color='red', linestyle='--', linewidth=1, label='Destabilizing threshold')

axes[-1].legend(fontsize=10, loc='upper right')
plt.suptitle('ddG of LoF Variants by Secondary Structure Element', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

sns.violinplot(
    data=lof_ss, x='ss_element', y='ddG',
    order=ss_order, palette=ss_palette,
    inner='box', cut=0, ax=ax,
)
sns.stripplot(
    data=lof_ss, x='ss_element', y='ddG',
    order=ss_order, color='black',
    size=2.5, alpha=0.5, jitter=True, ax=ax,
)

ax.axhline(1.665, color='red', linestyle='--', linewidth=1, label='Destabilizing threshold')
ax.set_xlabel('Secondary Structure Element', fontsize=12)
ax.set_ylabel('ddG (kcal/mol)', fontsize=12)
ax.set_title('ddG of LoF Variants by Secondary Structure Element\n(All Domains)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# SASA Analysis

## Read/merge files

In [ ]:
sasa_files = {
    'RING':'./Data/extra_data/SASA_data/BARD1_RING_1JM7_SASA.csv',
    'ARD':'./Data/extra_data/SASA_data/BARD1_ARD_3C5R_SASA.csv', 
    'BRCT':'./Data/extra_data/SASA_data/BARD1_BRCT_3FA2_SASA.csv'
}

In [ ]:
sasa_dfs = []
for domain in sasa_files:
    domain_sasa = pd.read_csv(sasa_files[domain])

    domain_sasa = domain_sasa[domain_sasa["resSeq"] > 10]

    AA3TO1 = { 
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D",
    "CYS": "C", "GLN": "Q", "GLU": "E", "GLY": "G",
    "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S",
    "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
    }

    domain_sasa['ref_aa']=domain_sasa['residue'].map(AA3TO1)
    domain_sasa['aa_id'] = domain_sasa['ref_aa'] + domain_sasa['resSeq'].astype(str)

    domain_sasa = domain_sasa.sort_values('relative_accessibility', ascending=False).drop_duplicates('aa_id').sort_index()

    domain_sasa['sasa_class'] = 'intermediate'

    domain_sasa.loc[domain_sasa['relative_accessibility'] >= 50, 'sasa_class'] = 'exposed'
    domain_sasa.loc[domain_sasa['relative_accessibility'] <= 20, 'sasa_class'] = 'buried'

    domain_sasa = domain_sasa.rename(columns={'resSeq': 'aa_pos'})

    domain_sasa = domain_sasa[['ref_aa', 'aa_pos', 'relative_accessibility', 'sasa_class']]

    sasa_dfs.append(domain_sasa)

sasa_df = pd.concat(sasa_dfs)

final_sasa_df = pd.merge(final_df, sasa_df, on=['ref_aa', 'aa_pos'], how='left')

final_sasa_df

## Violin plot by surface accessibility

In [ ]:
sasa_order=['buried', 'intermediate', 'exposed', 'functionally_normal']

filtered_final_sasa_df = final_sasa_df[
    (final_sasa_df['consequence'] == 'missense_variant') &
    (final_sasa_df['functional_consequence'].isin(['functionally_normal', 'functionally_abnormal'])) &
    (final_sasa_df['RNA_consequence'] == 'normal')
].copy()

plot_sasa_df = filtered_final_sasa_df.copy()

plot_sasa_df.loc[plot_sasa_df['functional_consequence']=='functionally_normal', 'sasa_class'] = 'functionally_normal'

fig, ax = plt.subplots(figsize=(6, 3))
sns.violinplot(
    data=plot_sasa_df,
    x='sasa_class',
    y='ddG',
    order=sasa_order,
    ax=ax
)

ax.set_xlabel('Surface Accessibility (LoF Only)')
ax.set_ylabel('ddG (kcal/mol)')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import ttest_ind

for cat1, cat2 in itertools.combinations(sasa_order, 2):
    g1 = plot_sasa_df[plot_sasa_df['sasa_class'] == cat1]['ddG'].dropna()
    g2 = plot_sasa_df[plot_sasa_df['sasa_class'] == cat2]['ddG'].dropna()
    t_stat, p_val = ttest_ind(g1, g2, equal_var=True)
    print(f'{cat1} vs {cat2}: t={t_stat:.4f}, p={p_val:.4e}  (n={len(g1)}, n={len(g2)})')

plot_sasa_df

## Proportion breakdown of things

In [ ]:
print(filtered_final_sasa_df[filtered_final_sasa_df['functional_consequence']=='functionally_abnormal'].value_counts(['sasa_class', 'ddG_class']))

# ThermoMPNN-I for Indels

In [ ]:
thermompnni_files = {
    'RING':'./Data/extra_data/ThermoMPNN_data/BARD1_RING_dels.csv',
    'ARD':'./Data/extra_data/ThermoMPNN_data/BARD1_ARD_dels.csv', 
    'BRCT':'./Data/extra_data/ThermoMPNN_data/BARD1_BRCT_dels.csv'
}

In [ ]:
domains = list(thermompnni_files.keys())

thermompnni_dfs = []

amino_acid_codes = {
    'A': 'Ala',  # Alanine
    'R': 'Arg',  # Arginine
    'N': 'Asn',  # Asparagine
    'D': 'Asp',  # Aspartic acid
    'C': 'Cys',  # Cysteine
    'E': 'Glu',  # Glutamic acid
    'Q': 'Gln',  # Glutamine
    'G': 'Gly',  # Glycine
    'H': 'His',  # Histidine
    'I': 'Ile',  # Isoleucine
    'L': 'Leu',  # Leucine
    'K': 'Lys',  # Lysine
    'M': 'Met',  # Methionine
    'F': 'Phe',  # Phenylalanine
    'P': 'Pro',  # Proline
    'S': 'Ser',  # Serine
    'T': 'Thr',  # Threonine
    'W': 'Trp',  # Tryptophan
    'Y': 'Tyr',  # Tyrosine
    'V': 'Val',  # Valine
}

for domain in domains:
    thermompnn_df = pd.read_csv(thermompnni_files[domain])

    hgvs_p_prefix='ENSP00000260947.4:p.' #Example in SGE scores ENSP00000260947.4:p.Asn425del
    thermompnn_df = thermompnn_df[thermompnn_df['pos_PDB']>10].copy()
    thermompnn_df['wtAA']=thermompnn_df['wtAA'].map(amino_acid_codes)

    thermompnn_df['hgvs_p']=hgvs_p_prefix + thermompnn_df['wtAA']+thermompnn_df['pos_PDB'].astype(str)+thermompnn_df['mutAA']

    thermompnn_df = thermompnn_df.rename(columns={'pos_PDB': 'aa_pos',
                                                    'wtAA': 'ref_aa',
                                                    'mutAA': 'alt_aa',
                                                    'ddG (kcal/mol)': 'ddG'})
    
    thermompnn_df['domain'] = domain
    thermompnn_df = thermompnn_df[['domain', 'ref_aa', 'alt_aa', 'aa_pos', 'ddG', 'hgvs_p']]
    thermompnni_dfs.append(thermompnn_df)



final_thermompnni_df = pd.concat(thermompnni_dfs)
final_thermompnni_df

In [ ]:
sge_del_df = raw_sge_df.dropna(subset='hgvs_p').copy()
sge_del_df=sge_del_df[(sge_del_df['var_type']=='3bp_del') & (~sge_del_df['hgvs_p'].str.contains('delins'))]

final_del_df = pd.merge(sge_del_df, final_thermompnni_df, on='hgvs_p', how='inner')


final_del_df

In [ ]:
final_del_df, _ = thermompnn_scorer(final_del_df, threshold=ddG_threshold, bpdel=True)

## Plots for indels

### Cumulator plot

In [ ]:
missub = final_del_df.sort_values(by="ddG")
gbobj = missub.groupby('functional_consequence')['ddG']
missub["cumulative_percentage"] = (gbobj.cumcount()  + 1)/ gbobj.transform('count') * 100.0


# draw the plot
chart = alt.Chart(missub, height=300, width=300).mark_line(size=5).encode(
    x=alt.X('ddG:Q', title='ddG').axis(labelFlush=False),
    y=alt.Y('cumulative_percentage:Q', title='Cumulative percentage of AA dels'),
    color = alt.Color('functional_consequence:N')
)

chart.display()

### Violin plot

In [ ]:
order = ['functionally_normal', 'indeterminate', 'functionally_abnormal']

fig, ax = plt.subplots(figsize=(8, 5))

sns.violinplot(
    data=final_del_df,
    x='functional_consequence',
    y='ddG',
    order=order,
    ax=ax
)

ax.set_xlabel('Functional Consequence')
ax.set_ylabel('ddG (kcal/mol)')
plt.tight_layout()
plt.show()

# Combined SNV and Del heatmap

In [ ]:
def domain_ddg_panel(df, domain_name, x_start, x_end, px_per_aa=4,
                     show_y_labels=True, show_legend=False, del_df=None):
    """ddG heatmap for one BARD1 domain with secondary structure cartoon on top.

    del_df : optional dataframe of single-residue deletion ddG values (final_del_df).
             When provided, a 'del' row is appended below the 20 missense AA rows.
    """

    ss_coords = {
        'RING': {
            'x':   [26,  34,  48,  61, 63,  68, 70,  74,  80,  97, 117],
            'x2':  [34,  48,  61,  63, 68,  70, 74,  80,  97, 117, 122],
            'color': ['white','blue','white','green','white','green',
                      'white','blue','white','blue','white'],
        },
        'ARD': {
            'x':  [425, 430, 439, 441, 450, 463, 471, 473, 483, 497, 504, 507, 516, 529, 534, 536],
            'x2': [430, 439, 441, 450, 463, 471, 473, 483, 497, 504, 507, 516, 529, 534, 536, 545],
            'color': ['white','blue','white','blue','white','blue','white','blue',
                      'white','blue','white','blue','white','blue','white','blue'],
        },
        'BRCT': {
            'x':  [568, 571, 574, 578, 592, 595, 597, 606, 608, 617, 626, 629, 631, 643,
                   655, 666, 676, 679, 687, 698, 700, 702, 712, 717, 735, 739, 750, 752, 755, 760, 770],
            'x2': [571, 574, 578, 592, 595, 597, 606, 608, 617, 626, 629, 631, 643, 655,
                   666, 676, 679, 687, 698, 700, 702, 712, 717, 735, 739, 750, 752, 755, 760, 770, 777],
            'color': ['white','green','white','blue','white','green','white','green',
                      'white','blue','white','green','blue','white','blue','white',
                      'green','white','blue','white','green','white','blue','white',
                      'green','white','green','white','green','blue','white'],
        },
    }

    dom_color = {'RING': '#B9DBF4', 'ARD': '#C8DBC8', 'BRCT': '#F6BF93'}[domain_name]
    ss = ss_coords[domain_name]
    width = (x_end - x_start) * px_per_aa
    n_aa  = x_end - x_start + 1

    # --- Cartoon strip ---
    ss_df  = pd.DataFrame({'x': ss['x'], 'x2': ss['x2'], 'y': 0, 'y2': 1, 'color': ss['color']})
    dom_df = pd.DataFrame({'x': [x_start], 'x2': [x_end], 'y': [1.15], 'y2': [2.15],
                           'color': [dom_color], 'label': [domain_name]})

    x_scale  = alt.Scale(domain=[x_start, x_end])
    y_cscale = alt.Scale(domain=[-0.15, 2.3])
    no_axis  = alt.Axis(labels=False, ticks=False, domain=False, title=None, grid=False)

    ss_rects = alt.Chart(ss_df).mark_rect(stroke='black', strokeWidth=1.5).encode(
        x =alt.X('x:Q',  scale=x_scale,  axis=no_axis),
        x2='x2:Q',
        y =alt.Y('y:Q',  scale=y_cscale, axis=no_axis),
        y2='y2:Q',
        color=alt.Color('color:N', scale=None, legend=None),
    )
    dom_rect = alt.Chart(dom_df).mark_rect(stroke='black', strokeWidth=2, opacity=0.85).encode(
        x =alt.X('x:Q',  scale=x_scale,  axis=no_axis),
        x2='x2:Q',
        y =alt.Y('y:Q',  scale=y_cscale, axis=no_axis),
        y2='y2:Q',
        color=alt.Color('color:N', scale=None, legend=None),
    )
    dom_text = alt.Chart(dom_df).mark_text(
        fontWeight='bold', fontSize=14, color='black', align='center', baseline='middle'
    ).encode(
        x   =alt.X('x_mid:Q', scale=x_scale,  axis=no_axis),
        y   =alt.Y('y_mid:Q', scale=y_cscale, axis=no_axis),
        text='label:N',
    ).transform_calculate(
        x_mid='(datum.x + datum.x2) / 2',
        y_mid='(datum.y + datum.y2) / 2',
    )

    cartoon = alt.layer(ss_rects, dom_rect, dom_text).properties(width=width, height=60)

    # --- ddG heatmap data ---
    aa_order = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']

    # Pad with NaN sentinels for any AA absent in this domain so all 20 rows always render.
    dom_data = df[df['domain'] == domain_name].copy()
    missing_aas = [aa for aa in aa_order if aa not in dom_data['alt_aa'].values]
    if missing_aas:
        sentinels = pd.DataFrame({
            'alt_aa': missing_aas, 'aa_pos': x_start,
            'ddG': float('nan'), 'amino_acid_change': '', 'score': float('nan'),
        })
        dom_data = pd.concat([dom_data, sentinels], ignore_index=True)

    # Append single-residue deletion row when del_df is supplied.
    if del_df is not None:
        del_rows = (
            del_df[del_df['domain'] == domain_name]
            [['aa_pos', 'ddG', 'domain', 'alt_aa']]
            .drop_duplicates(subset=['aa_pos'])
            .copy()
        )
        # Drop any positions outside this panel's x-axis range.
        del_rows = del_rows[(del_rows['aa_pos'] >= x_start) & (del_rows['aa_pos'] <= x_end)]
        del_rows['amino_acid_change'] = del_rows['aa_pos'].astype(str) + 'del'
        del_rows['score'] = float('nan')
        dom_data = pd.concat([dom_data, del_rows], ignore_index=True)
        full_order = aa_order + ['del']
    else:
        full_order = aa_order

    if show_y_labels:
        y_ax    = alt.Axis(labelFontSize=y_label_size, titleFontSize=y_title_size)
        y_title = 'Amino Acid Substitution'
    else:
        y_ax    = alt.Axis(labels=False, ticks=False, title=None, domain=False)
        y_title = ''

    # Only one panel carries the legend — Altair drops it when all panels are identical.
    ddg_legend = alt.Legend(titleFontSize=legend_title_size, labelFontSize=legend_label_size) if show_legend else None

    heat = alt.Chart(dom_data).mark_rect().encode(
        x=alt.X('aa_pos:Q',
                title='Amino Acid Position',
                axis=alt.Axis(labelFontSize=x_label_size, titleFontSize=x_title_size,
                              values=list(range(0, 800, 25))),
                scale=x_scale,
                bin=alt.Bin(maxbins=n_aa, minstep=1)),
        y=alt.Y('alt_aa',
                title=y_title,
                axis=y_ax,
                sort=full_order),
        color=alt.Color('ddG',
                        title='ddG (kcal/mol)',
                        scale=alt.Scale(domain=[0, 3], clamp=True, scheme='bluepurple'),
                        legend=ddg_legend),
        tooltip=['amino_acid_change', 'aa_pos', 'alt_aa', 'ddG'],
    ).properties(height=20 * len(full_order), width=width)

    return alt.vconcat(cartoon, heat, spacing=2)


def faceted_ddg_heatmap(df, px_per_aa=4, del_df=None):
    """ddG heatmap faceted by BARD1 domain (RING / ARD / BRCT) with secondary structure cartoons.

    del_df : optional final_del_df — adds a 'del' row to each domain panel.
    """
    domains = [('RING', 26, 122), ('ARD', 425, 545), ('BRCT', 568, 777)]

    panels = [
        domain_ddg_panel(df, name, start, end, px_per_aa,
                         show_y_labels=(i == 0),
                         show_legend=(i == 2),
                         del_df=del_df)
        for i, (name, start, end) in enumerate(domains)
    ]

    return (
        alt.hconcat(*panels, spacing=20)
        .resolve_scale(color='independent')
        .configure_axis(grid=False)
        .configure_view(stroke=None)
    )

In [ ]:
faceted_ddg_heatmap(final_df[final_df['functional_consequence']=='functionally_abnormal'], del_df=final_del_df[final_del_df['functional_consequence'] == 'functionally_abnormal'])
